In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "README.md").is_file() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "README.md").is_file():
    raise FileNotFoundError("Run this notebook from the project directory or one of its subdirectories.")


In [ ]:
# ============================================================
# preprocess_v2.py
#
# V2 BASE PREPROCESSING
#
# INPUT
#   train_atmos.csv
#   train_wave.csv
#
# OUTPUT
#   train.csv
#   train_final_base_v2.csv
#
# 핵심 전략
# ------------------------------------------------------------
# 1. 기존 raw merge 구조 유지
# 2. 명백한 물리 범위 이상치만 제거
# 3. 범용 interpolate(limit=2) 제거
# 4. 방향값 일반 linear interpolation 제거
# 5. station별 Hs 전략
#
#    G-ORS:
#       <=3h linear
#       >3h NaN 유지
#
#    I-ORS:
#       <=3h linear
#       >24h G-ORS lag transfer
#
#    S-ORS:
#       <=3h linear
#       >3h NaN 유지
#
# 6. Tp <=3h linear
# 7. Hmax <- station별 Hmax/Hs robust ratio
# 8. Atmos <=1h linear
# 9. wdir/wvdir isolated 1-step circular
# 10. original/imputation flag 보존
# 11. physics feature 생성 X
# ============================================================


from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression


# ============================================================
# PATH
# ============================================================

RAW_ATMOS_PATH = Path(PROJECT_ROOT / "data" / "raw" / "train_atmos.csv")
RAW_WAVE_PATH = Path(PROJECT_ROOT / "data" / "raw" / "train_wave.csv")

MERGED_PATH = Path(PROJECT_ROOT / "data" / "processed" / "train.csv")

FINAL_BASE_PATH = Path(
    PROJECT_ROOT / "data" / "processed" / "train_v2.csv"
)


# ============================================================
# CONFIG
# ============================================================

TIME_COL = "time"
STATION_COL = "station"

STEP_MINUTES = 10
STEPS_PER_HOUR = 6


BASE_COLUMNS = [
    "hs",
    "tp",
    "hmax",
    "wvdir",
    "wspd",
    "gust",
    "wdir",
    "airt",
    "relh",
    "caph",
]


CONTINUOUS_COLUMNS = [
    "hs",
    "tp",
    "hmax",
    "wspd",
    "gust",
    "airt",
    "relh",
    "caph",
]


DIRECTION_COLUMNS = [
    "wvdir",
    "wdir",
]


# ------------------------------------------------------------
# Hs
# ------------------------------------------------------------

HS_SHORT_MAX_STEPS = 18      # 3h

I_LONG_MIN_STEPS = 145       # >24h

G_TO_I_LAG_STEPS = 17        # 2h50m


# ------------------------------------------------------------
# Tp
# ------------------------------------------------------------

TP_SHORT_MAX_STEPS = 18      # 3h


# ------------------------------------------------------------
# Atmos
# ------------------------------------------------------------

ATMOS_SHORT_MAX_STEPS = 6    # 1h


# ============================================================
# 1. RAW MERGE
# ============================================================

def create_merged_train():

    atmos = pd.read_csv(
        RAW_ATMOS_PATH
    )

    wave = pd.read_csv(
        RAW_WAVE_PATH
    )


    atmos[TIME_COL] = pd.to_datetime(
        atmos[TIME_COL]
    )

    wave[TIME_COL] = pd.to_datetime(
        wave[TIME_COL]
    )


    train = pd.merge(
        atmos,
        wave,
        on=[
            STATION_COL,
            TIME_COL,
        ],
        how="outer",
    )


    train = (
        train
        .sort_values(
            [
                STATION_COL,
                TIME_COL,
            ]
        )
        .reset_index(drop=True)
    )


    preferred_order = [
        "case_id",
        STATION_COL,
        "step_minute",
        TIME_COL,
        "hs",
        "tp",
        "hmax",
        "wvdir",
        "wspd",
        "gust",
        "wdir",
        "airt",
        "relh",
        "caph",
    ]


    existing = [
        c
        for c in preferred_order
        if c in train.columns
    ]

    remaining = [
        c
        for c in train.columns
        if c not in existing
    ]


    train = train[
        existing + remaining
    ]


    train.to_csv(
        MERGED_PATH,
        index=False
    )


    print(
        "Merged:",
        train.shape
    )


    return train


# ============================================================
# 2. ORIGINAL FLAGS
#
# 반드시 어떠한 cleaning보다 먼저 생성
# ============================================================

def add_original_flags(df):

    df = df.copy()


    for col in BASE_COLUMNS:

        df[
            f"{col}_original_observed"
        ] = (
            df[col]
            .notna()
            .astype(np.int8)
        )


        df[
            f"{col}_imputed"
        ] = np.int8(0)


    # Hs는 출처를 상세 기록
    df["hs_fill_method"] = np.where(
        df["hs"].notna(),
        "observed",
        "missing",
    )


    return df


# ============================================================
# 3. CONSERVATIVE ANOMALY CLEANING
#
# 기존 V1의 공격적 규칙은 제거
#
# 삭제한 것:
# - frozen 3point
# - hs diff > 1m
# - 모든 변수 interpolate(limit=2)
# ============================================================

def conservative_anomaly_cleaning(
    df
):

    df = df.copy()


    # --------------------------------------------------------
    # Wave
    # --------------------------------------------------------

    df.loc[
        df["hs"] <= 0,
        "hs"
    ] = np.nan


    df.loc[
        df["tp"] <= 0,
        "tp"
    ] = np.nan


    df.loc[
        df["hmax"] <= 0,
        "hmax"
    ] = np.nan


    # --------------------------------------------------------
    # Wind
    # --------------------------------------------------------

    df.loc[
        df["wspd"] < 0,
        "wspd"
    ] = np.nan


    df.loc[
        df["gust"] < 0,
        "gust"
    ] = np.nan


    # --------------------------------------------------------
    # Atmos
    # --------------------------------------------------------

    df.loc[
        (
            df["caph"] < 950
        )
        |
        (
            df["caph"] > 1050
        ),
        "caph"
    ] = np.nan


    df.loc[
        (
            df["relh"] < 0
        )
        |
        (
            df["relh"] > 100
        ),
        "relh"
    ] = np.nan


    # --------------------------------------------------------
    # Direction normalization
    # --------------------------------------------------------

    for col in DIRECTION_COLUMNS:

        mask = (
            df[col]
            .notna()
        )

        df.loc[
            mask,
            col
        ] = (
            df.loc[
                mask,
                col
            ]
            %
            360
        )


    # --------------------------------------------------------
    # Hmax < Hs
    #
    # 강제로 max() 하지 않고
    # Hmax를 invalid 처리
    # --------------------------------------------------------

    invalid_hmax = (
        df["hmax"].notna()
        &
        df["hs"].notna()
        &
        (
            df["hmax"]
            <
            df["hs"]
        )
    )


    df.loc[
        invalid_hmax,
        "hmax"
    ] = np.nan


    # --------------------------------------------------------
    # gust < wspd
    #
    # 강제로 np.maximum() 하지 않음
    # gust 측 값을 invalid 처리
    # --------------------------------------------------------

    invalid_gust = (
        df["gust"].notna()
        &
        df["wspd"].notna()
        &
        (
            df["gust"]
            <
            df["wspd"]
        )
    )


    df.loc[
        invalid_gust,
        "gust"
    ] = np.nan


    print(
        "\n===== ANOMALY CLEANING ====="
    )

    print(
        "Hmax < Hs removed:",
        int(
            invalid_hmax.sum()
        )
    )

    print(
        "Gust < Wspd removed:",
        int(
            invalid_gust.sum()
        )
    )


    return df


# ============================================================
# 4. GAP UTILITY
# ============================================================

def get_missing_runs(
    series
):

    values = (
        series
        .to_numpy()
    )


    missing = pd.isna(
        values
    )


    runs = []

    pos = 0
    n = len(values)


    while pos < n:

        if not missing[pos]:

            pos += 1
            continue


        start = pos


        while (
            pos < n
            and
            missing[pos]
        ):

            pos += 1


        end = pos - 1


        runs.append(
            (
                start,
                end,
                end - start + 1,
            )
        )


    return runs


# ============================================================
# 5. GENERIC SHORT LINEAR INTERPOLATION
#
# gap 전체 길이가 threshold 이하일 때만
# 양쪽 boundary가 존재해야 함
# ============================================================

def fill_short_linear(
    df,
    col,
    max_steps,
    stations=None,
    method_name=None,
):

    total = 0


    if stations is None:

        stations = (
            df[STATION_COL]
            .dropna()
            .unique()
        )


    flag_col = (
        f"{col}_imputed"
    )


    for station in stations:

        index = df.index[
            df[STATION_COL]
            ==
            station
        ]


        s = (
            df.loc[
                index,
                col
            ]
            .copy()
        )


        runs = (
            get_missing_runs(
                s
            )
        )


        for (
            start,
            end,
            gap_len
        ) in runs:


            if (
                gap_len
                >
                max_steps
            ):

                continue


            left = (
                start - 1
            )

            right = (
                end + 1
            )


            if (
                left < 0
                or
                right >= len(s)
            ):

                continue


            left_value = (
                s.iloc[
                    left
                ]
            )

            right_value = (
                s.iloc[
                    right
                ]
            )


            if (
                pd.isna(
                    left_value
                )
                or
                pd.isna(
                    right_value
                )
            ):

                continue


            fill_values = np.linspace(
                left_value,
                right_value,
                gap_len + 2,
            )[1:-1]


            fill_index = (
                index[
                    start:
                    end + 1
                ]
            )


            df.loc[
                fill_index,
                col
            ] = fill_values


            df.loc[
                fill_index,
                flag_col
            ] = 1


            if (
                col == "hs"
                and
                method_name is not None
            ):

                df.loc[
                    fill_index,
                    "hs_fill_method"
                ] = method_name


            # local copy 갱신
            s.iloc[
                start:
                end + 1
            ] = fill_values


            total += gap_len


    return total


# ============================================================
# 6. EXACT 1-STEP LINEAR
#
# 모든 continuous variable 공통
# ============================================================

def fill_exact_single_linear(
    df
):

    print(
        "\n===== EXACT 1-STEP LINEAR ====="
    )


    summary = {}


    for col in CONTINUOUS_COLUMNS:

        method = (
            "single_linear"
            if col == "hs"
            else None
        )


        n = fill_short_linear(
            df=df,
            col=col,
            max_steps=1,
            method_name=method,
        )


        summary[col] = n


    print(
        pd.Series(
            summary,
            name="filled"
        )
    )


    return df


# ============================================================
# 7. CIRCULAR INTERPOLATION
#
# exact isolated 1-step only
# ============================================================

def circular_midpoint(
    a,
    b
):

    a = np.deg2rad(a)
    b = np.deg2rad(b)


    x = (
        np.cos(a)
        +
        np.cos(b)
    )

    y = (
        np.sin(a)
        +
        np.sin(b)
    )


    return (
        np.rad2deg(
            np.arctan2(
                y,
                x
            )
        )
        %
        360
    )


def fill_single_direction(
    df,
    col
):

    total = 0


    for station in (
        df[STATION_COL]
        .dropna()
        .unique()
    ):

        index = df.index[
            df[STATION_COL]
            ==
            station
        ]


        values = (
            df.loc[
                index,
                col
            ]
            .to_numpy(
                dtype=float
            )
        )


        original_missing = (
            np.isnan(
                values
            )
        )


        for i in range(
            1,
            len(values) - 1
        ):

            # 반드시 원래 isolated 1-step
            if not original_missing[i]:

                continue


            if (
                original_missing[
                    i - 1
                ]
                or
                original_missing[
                    i + 1
                ]
            ):

                continue


            if (
                not np.isfinite(
                    values[
                        i - 1
                    ]
                )
                or
                not np.isfinite(
                    values[
                        i + 1
                    ]
                )
            ):

                continue


            value = (
                circular_midpoint(
                    values[
                        i - 1
                    ],
                    values[
                        i + 1
                    ],
                )
            )


            global_index = (
                index[i]
            )


            df.loc[
                global_index,
                col
            ] = value


            df.loc[
                global_index,
                f"{col}_imputed"
            ] = 1


            total += 1


    return total


def fill_direction_gaps(
    df
):

    print(
        "\n===== DIRECTION 1-STEP ====="
    )


    for col in DIRECTION_COLUMNS:

        n = fill_single_direction(
            df,
            col
        )

        print(
            col,
            ":",
            n
        )


    return df


# ============================================================
# 8. STATION-SPECIFIC HS SHORT GAPS
#
# exact 1-step은 앞 단계에서 이미 채워짐
#
# 여기서는 남은 gap 중 <=3h
# ============================================================

def fill_station_hs_short_gaps(
    df
):

    print(
        "\n===== STATION-SPECIFIC HS ====="
    )


    # --------------------------------------------------------
    # G
    # --------------------------------------------------------

    g_n = fill_short_linear(
        df=df,
        col="hs",
        max_steps=HS_SHORT_MAX_STEPS,
        stations=["G-ORS"],
        method_name="short_linear",
    )


    # --------------------------------------------------------
    # I
    # --------------------------------------------------------

    i_n = fill_short_linear(
        df=df,
        col="hs",
        max_steps=HS_SHORT_MAX_STEPS,
        stations=["I-ORS"],
        method_name="short_linear",
    )


    # --------------------------------------------------------
    # S
    # --------------------------------------------------------

    s_n = fill_short_linear(
        df=df,
        col="hs",
        max_steps=HS_SHORT_MAX_STEPS,
        stations=["S-ORS"],
        method_name="short_linear",
    )


    print(
        "G-ORS short Hs:",
        g_n
    )

    print(
        "I-ORS short Hs:",
        i_n
    )

    print(
        "S-ORS short Hs:",
        s_n
    )


    return df


# ============================================================
# 9. I-ORS LONG GAP
#
# I(t) <- a * G(t - 17step) + b
#
# calibration:
# - gap 시작 이전만
# - original observed I 사용
# - original observed G 사용
#
# source prediction:
# - G short interpolation 값은 사용 가능
#   (과거 input reconstruction 목적)
# ============================================================

def fill_i_long_gap_from_g(
    df
):

    print(
        "\n===== I-ORS LONG GAP <- G-ORS ====="
    )


    # --------------------------------------------------------
    # 현재 Hs wide
    # --------------------------------------------------------

    hs_wide = (
        df
        .pivot(
            index=TIME_COL,
            columns=STATION_COL,
            values="hs",
        )
        .sort_index()
    )


    g_lag = (
        hs_wide[
            "G-ORS"
        ]
        .shift(
            G_TO_I_LAG_STEPS
        )
    )


    # --------------------------------------------------------
    # original observation wide
    # --------------------------------------------------------

    obs_wide = (
        df
        .pivot(
            index=TIME_COL,
            columns=STATION_COL,
            values="hs_original_observed",
        )
        .sort_index()
    )


    g_obs_lag = (
        obs_wide[
            "G-ORS"
        ]
        .shift(
            G_TO_I_LAG_STEPS
        )
    )


    i_mask = (
        df[STATION_COL]
        ==
        "I-ORS"
    )


    i_df = (
        df.loc[
            i_mask,
            [
                TIME_COL,
                "hs",
            ]
        ]
        .sort_values(
            TIME_COL
        )
        .copy()
    )


    i_indices = (
        i_df.index
        .to_numpy()
    )


    runs = get_missing_runs(
        i_df["hs"]
    )


    long_runs = [
        run
        for run in runs
        if run[2]
        >=
        I_LONG_MIN_STEPS
    ]


    print(
        "Long I gaps:",
        len(
            long_runs
        )
    )


    total_fill = 0


    for number, (
        start,
        end,
        gap_len
    ) in enumerate(
        long_runs,
        1
    ):


        gap_start = (
            i_df.iloc[
                start
            ][
                TIME_COL
            ]
        )


        gap_end = (
            i_df.iloc[
                end
            ][
                TIME_COL
            ]
        )


        # ====================================================
        # CALIBRATION
        #
        # 반드시 gap 이전만
        # ====================================================

        calibration = (
            pd.DataFrame({

                "I":
                    hs_wide[
                        "I-ORS"
                    ],

                "G_lag":
                    g_lag,

                "I_obs":
                    obs_wide[
                        "I-ORS"
                    ],

                "G_obs_lag":
                    g_obs_lag,
            })
        )


        calibration = calibration[
            calibration.index
            <
            gap_start
        ]


        calibration = calibration[
            (
                calibration[
                    "I_obs"
                ]
                ==
                1
            )
            &
            (
                calibration[
                    "G_obs_lag"
                ]
                ==
                1
            )
        ]


        calibration = (
            calibration[
                [
                    "I",
                    "G_lag",
                ]
            ]
            .dropna()
        )


        print(
            f"\nGap {number}:",
            gap_start,
            "->",
            gap_end,
        )


        print(
            "steps:",
            gap_len,
            "| days:",
            round(
                gap_len
                /
                144,
                2
            )
        )


        print(
            "calibration:",
            len(
                calibration
            )
        )


        if (
            len(
                calibration
            )
            <
            1000
        ):

            print(
                "SKIP - calibration 부족"
            )

            continue


        model = LinearRegression()


        model.fit(
            calibration[
                [
                    "G_lag"
                ]
            ],
            calibration[
                "I"
            ],
        )


        print(
            "coef:",
            round(
                model.coef_[0],
                5
            )
        )


        print(
            "intercept:",
            round(
                model.intercept_,
                5
            )
        )


        # ====================================================
        # PREDICTION
        # ====================================================

        gap_times = (
            i_df.iloc[
                start:
                end + 1
            ][
                TIME_COL
            ]
        )


        source = (
            g_lag
            .reindex(
                gap_times
            )
        )


        valid_source = (
            source
            .notna()
            .to_numpy()
        )


        print(
            "source coverage:",
            f"{valid_source.mean() * 100:.2f}%"
        )


        if (
            valid_source.sum()
            ==
            0
        ):

            continue


        pred = np.full(
            gap_len,
            np.nan,
            dtype=float,
        )


        pred[
            valid_source
        ] = (
            model.predict(
                source[
                    valid_source
                ]
                .to_numpy()
                .reshape(
                    -1,
                    1
                )
            )
        )


        pred = np.where(
            np.isfinite(
                pred
            ),
            np.clip(
                pred,
                0.01,
                None
            ),
            np.nan,
        )


        global_index = (
            i_indices[
                start:
                end + 1
            ]
        )


        currently_missing = (
            df.loc[
                global_index,
                "hs"
            ]
            .isna()
            .to_numpy()
        )


        fillable = (
            valid_source
            &
            currently_missing
            &
            np.isfinite(
                pred
            )
        )


        fill_index = (
            global_index[
                fillable
            ]
        )


        df.loc[
            fill_index,
            "hs"
        ] = (
            pred[
                fillable
            ]
        )


        df.loc[
            fill_index,
            "hs_imputed"
        ] = 1


        df.loc[
            fill_index,
            "hs_fill_method"
        ] = (
            "G_lag_transfer"
        )


        total_fill += len(
            fill_index
        )


    print(
        "\nI transfer total:",
        total_fill
    )


    return df


# ============================================================
# 10. TP SHORT GAP
#
# <=3h
# ============================================================

def fill_tp_short_gaps(
    df
):

    print(
        "\n===== TP <=3H ====="
    )


    n = fill_short_linear(
        df=df,
        col="tp",
        max_steps=TP_SHORT_MAX_STEPS,
    )


    print(
        "Tp filled:",
        n
    )


    return df


# ============================================================
# 11. HMAX RECONSTRUCTION
#
# station별 Hmax/Hs median
#
# ratio calibration에는
# original observed pair만 사용
# ============================================================

def reconstruct_hmax(
    df
):

    print(
        "\n===== HMAX <- HS ====="
    )


    calibration = df[
        (
            df[
                "hs_original_observed"
            ]
            ==
            1
        )
        &
        (
            df[
                "hmax_original_observed"
            ]
            ==
            1
        )
        &
        df["hs"].notna()
        &
        df["hmax"].notna()
        &
        (
            df["hs"]
            >
            0
        )
    ].copy()


    calibration[
        "_ratio"
    ] = (
        calibration[
            "hmax"
        ]
        /
        calibration[
            "hs"
        ]
    )


    # 이상 ratio 제외
    calibration = calibration[
        calibration[
            "_ratio"
        ]
        .between(
            1.0,
            3.0
        )
    ]


    station_ratio = (
        calibration
        .groupby(
            STATION_COL
        )[
            "_ratio"
        ]
        .median()
    )


    print(
        "\nHmax/Hs ratio:"
    )

    print(
        station_ratio
    )


    total = 0


    for station, ratio in (
        station_ratio.items()
    ):


        mask = (
            (
                df[STATION_COL]
                ==
                station
            )
            &
            df["hmax"].isna()
            &
            df["hs"].notna()
        )


        n = int(
            mask.sum()
        )


        df.loc[
            mask,
            "hmax"
        ] = (
            df.loc[
                mask,
                "hs"
            ]
            *
            ratio
        )


        df.loc[
            mask,
            "hmax_imputed"
        ] = 1


        total += n


    print(
        "\nHmax filled:",
        total
    )


    return df


# ============================================================
# 12. ATMOS SHORT GAP
#
# <=1h
#
# wdir는 circular에서 이미 처리했으므로 제외
# ============================================================

def fill_short_atmos(
    df
):

    print(
        "\n===== ATMOS <=1H ====="
    )


    atmos_columns = [
        "wspd",
        "gust",
        "airt",
        "relh",
        "caph",
    ]


    summary = {}


    for col in (
        atmos_columns
    ):

        n = fill_short_linear(
            df=df,
            col=col,
            max_steps=ATMOS_SHORT_MAX_STEPS,
        )


        summary[col] = n


    print(
        pd.Series(
            summary,
            name="filled"
        )
    )


    return df


# ============================================================
# 13. FINAL SANITY CHECK
# ============================================================

def final_sanity(
    df
):

    df = df.copy()


    # --------------------------------------------------------
    # Base physical bounds
    # --------------------------------------------------------

    for col in [
        "hs",
        "tp",
        "hmax",
    ]:

        df.loc[
            df[col]
            <=
            0,
            col
        ] = np.nan


    df.loc[
        df["wspd"]
        <
        0,
        "wspd"
    ] = np.nan


    df.loc[
        df["gust"]
        <
        0,
        "gust"
    ] = np.nan


    df.loc[
        ~df["relh"].between(
            0,
            100,
        ),
        "relh"
    ] = np.nan


    df.loc[
        ~df["caph"].between(
            950,
            1050,
        ),
        "caph"
    ] = np.nan


    for col in DIRECTION_COLUMNS:

        df[col] = (
            df[col]
            %
            360
        )


    # --------------------------------------------------------
    # hmax physical relation
    # --------------------------------------------------------

    bad_hmax = (
        df["hmax"].notna()
        &
        df["hs"].notna()
        &
        (
            df["hmax"]
            <
            df["hs"]
        )
    )


    print(
        "\nHmax < Hs violations:",
        int(
            bad_hmax.sum()
        )
    )


    return df


# ============================================================
# 14. DIAGNOSTICS
# ============================================================

def print_diagnostics(
    df
):

    print(
        "\n"
        +
        "=" * 80
    )

    print(
        "FINAL V2 DIAGNOSTICS"
    )

    print(
        "=" * 80
    )


    # --------------------------------------------------------
    # Missing
    # --------------------------------------------------------

    missing = pd.DataFrame({

        "missing":
            df[
                BASE_COLUMNS
            ]
            .isna()
            .sum(),

        "missing_pct":
            (
                df[
                    BASE_COLUMNS
                ]
                .isna()
                .mean()
                *
                100
            ),
    })


    print(
        "\n===== FINAL MISSING ====="
    )

    print(
        missing
        .sort_values(
            "missing_pct",
            ascending=False
        )
        .round(3)
    )


    # --------------------------------------------------------
    # Hs by station
    # --------------------------------------------------------

    print(
        "\n===== HS BY STATION ====="
    )


    hs_station = (
        df
        .groupby(
            STATION_COL
        )
        .agg(

            rows=(
                "hs",
                "size"
            ),

            original_hs=(
                "hs_original_observed",
                "sum"
            ),

            hs_available=(
                "hs",
                lambda s:
                s.notna().sum()
            ),

            hs_missing=(
                "hs",
                lambda s:
                s.isna().sum()
            ),

            hs_imputed=(
                "hs_imputed",
                "sum"
            ),
        )
    )


    print(
        hs_station
    )


    # --------------------------------------------------------
    # Hs source
    # --------------------------------------------------------

    print(
        "\n===== HS FILL METHOD ====="
    )


    print(
        pd.crosstab(
            df[
                STATION_COL
            ],
            df[
                "hs_fill_method"
            ],
        )
    )


    # --------------------------------------------------------
    # Imputation
    # --------------------------------------------------------

    print(
        "\n===== IMPUTED COUNTS ====="
    )


    flag_cols = [
        c
        for c in df.columns
        if c.endswith(
            "_imputed"
        )
    ]


    print(
        df[
            flag_cols
        ]
        .sum()
        .sort_values(
            ascending=False
        )
    )


    # --------------------------------------------------------
    # Remaining long Hs gaps
    # --------------------------------------------------------

    print(
        "\n===== REMAINING HS LONG GAPS ====="
    )


    records = []


    for station, group in (
        df.groupby(
            STATION_COL
        )
    ):

        group = (
            group
            .sort_values(
                TIME_COL
            )
        )


        runs = get_missing_runs(
            group["hs"]
        )


        for (
            start,
            end,
            n
        ) in runs:

            if n > 18:

                records.append({

                    "station":
                        station,

                    "start":
                        group.iloc[
                            start
                        ][
                            TIME_COL
                        ],

                    "end":
                        group.iloc[
                            end
                        ][
                            TIME_COL
                        ],

                    "steps":
                        n,

                    "hours":
                        n
                        /
                        6,

                    "days":
                        n
                        /
                        144,
                })


    long_gaps = pd.DataFrame(
        records
    )


    if len(
        long_gaps
    ):

        print(
            long_gaps
            .sort_values(
                "steps",
                ascending=False
            )
            .head(
                30
            )
        )

    else:

        print(
            "No >3h Hs gaps"
        )


# ============================================================
# 15. COLUMN ORDER
# ============================================================

def arrange_columns(
    df
):

    identity = [
        c
        for c in [
            "case_id",
            STATION_COL,
            "step_minute",
            TIME_COL,
        ]
        if c in df.columns
    ]


    base = [
        c
        for c in BASE_COLUMNS
        if c in df.columns
    ]


    observed = [
        f"{c}_original_observed"
        for c in BASE_COLUMNS
        if (
            f"{c}_original_observed"
            in df.columns
        )
    ]


    imputed = [
        f"{c}_imputed"
        for c in BASE_COLUMNS
        if (
            f"{c}_imputed"
            in df.columns
        )
    ]


    extra = [
        "hs_fill_method"
    ]


    ordered = (
        identity
        +
        base
        +
        observed
        +
        imputed
        +
        extra
    )


    remaining = [
        c
        for c in df.columns
        if c not in ordered
    ]


    return df[
        ordered
        +
        remaining
    ]


# ============================================================
# 16. MAIN
# ============================================================

def main():

    # --------------------------------------------------------
    # RAW
    # --------------------------------------------------------

    df = create_merged_train()


    print(
        "\nInitial shape:",
        df.shape
    )


    # --------------------------------------------------------
    # 가장 먼저 원본 여부 저장
    # --------------------------------------------------------

    df = add_original_flags(
        df
    )


    # --------------------------------------------------------
    # conservative anomaly
    # --------------------------------------------------------

    df = conservative_anomaly_cleaning(
        df
    )


    # --------------------------------------------------------
    # exact 1-step continuous
    # --------------------------------------------------------

    df = fill_exact_single_linear(
        df
    )


    # --------------------------------------------------------
    # direction isolated 1-step
    # --------------------------------------------------------

    df = fill_direction_gaps(
        df
    )


    # --------------------------------------------------------
    # station-specific Hs
    # --------------------------------------------------------

    df = fill_station_hs_short_gaps(
        df
    )


    # --------------------------------------------------------
    # I long outage <- G
    # --------------------------------------------------------

    df = fill_i_long_gap_from_g(
        df
    )


    # --------------------------------------------------------
    # Tp <=3h
    # --------------------------------------------------------

    df = fill_tp_short_gaps(
        df
    )


    # --------------------------------------------------------
    # Hmax reconstruction
    # --------------------------------------------------------

    df = reconstruct_hmax(
        df
    )


    # --------------------------------------------------------
    # Atmos <=1h
    # --------------------------------------------------------

    df = fill_short_atmos(
        df
    )


    # --------------------------------------------------------
    # sanity
    # --------------------------------------------------------

    df = final_sanity(
        df
    )


    # --------------------------------------------------------
    # sort
    # --------------------------------------------------------

    df = (
        df
        .sort_values(
            [
                STATION_COL,
                TIME_COL,
            ]
        )
        .reset_index(
            drop=True
        )
    )


    df = arrange_columns(
        df
    )


    # --------------------------------------------------------
    # diagnostics
    # --------------------------------------------------------

    print_diagnostics(
        df
    )


    # --------------------------------------------------------
    # SAVE ONCE
    # --------------------------------------------------------

    df.to_csv(
        FINAL_BASE_PATH,
        index=False
    )


    print(
        "\n"
        +
        "=" * 80
    )

    print(
        "SAVED"
    )

    print(
        "=" * 80
    )


    print(
        FINAL_BASE_PATH
    )


    print(
        "shape:",
        df.shape
    )


    print(
        "\nPhysics features included:",
        False
    )


    return df


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    final_v2 = main()

Merged: (183600, 12)

Initial shape: (183600, 12)

===== ANOMALY CLEANING =====
Hmax < Hs removed: 0
Gust < Wspd removed: 5561

===== EXACT 1-STEP LINEAR =====
hs      63752
tp      62622
hmax    63904
wspd       39
gust     3736
airt       59
relh       76
caph       43
Name: filled, dtype: int64

===== DIRECTION 1-STEP =====
wvdir : 64451
wdir : 30

===== STATION-SPECIFIC HS =====
G-ORS short Hs: 691
I-ORS short Hs: 241
S-ORS short Hs: 496

===== I-ORS LONG GAP <- G-ORS =====
Long I gaps: 1

Gap 1: 2024-07-24 22:00:00+09:00 -> 2024-11-14 23:40:00+09:00
steps: 8142 | days: 56.54
calibration: 0
SKIP - calibration 부족

I transfer total: 0

===== TP <=3H =====
Tp filled: 4336

===== HMAX <- HS =====

Hmax/Hs ratio:
station
G-ORS    1.645570
I-ORS    1.632812
S-ORS    1.645570
Name: _ratio, dtype: float64

Hmax filled: 1089

===== ATMOS <=1H =====
wspd     160
gust    1975
airt     135
relh     354
caph      71
Name: filled, dtype: int64

Hmax < Hs violations: 5

FINAL V2 DIAGNOSTICS

====

In [3]:
# ============================================================
# V2 PREPROCESSING VALIDATION
# train_final_v2.csv 검증
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path


PATH = Path(PROJECT_ROOT / "data" / "processed" / "train_final_v2.csv")

df = pd.read_csv(
    PATH,
    parse_dates=["time"]
)

df = (
    df
    .sort_values(["station", "time"])
    .reset_index(drop=True)
)


BASE_COLS = [
    "hs",
    "tp",
    "hmax",
    "wvdir",
    "wspd",
    "gust",
    "wdir",
    "airt",
    "relh",
    "caph",
]


print("=" * 80)
print("1. BASIC")
print("=" * 80)

print("shape:", df.shape)
print("stations:", df["station"].unique())
print("time:", df["time"].min(), "~", df["time"].max())


# ============================================================
# 2. MISSING SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("2. FINAL MISSING")
print("=" * 80)

missing = pd.DataFrame({
    "missing_n":
        df[BASE_COLS]
        .isna()
        .sum(),

    "missing_pct":
        df[BASE_COLS]
        .isna()
        .mean()
        * 100,
})

display(
    missing
    .sort_values(
        "missing_pct",
        ascending=False
    )
    .round(3)
)


# ============================================================
# 3. HS BY STATION
# ============================================================

print("\n" + "=" * 80)
print("3. HS BY STATION")
print("=" * 80)

hs_station = (
    df.groupby("station")
    .agg(
        rows=("hs", "size"),

        original_hs=(
            "hs_original_observed",
            "sum"
        ),

        hs_available=(
            "hs",
            lambda x: x.notna().sum()
        ),

        hs_missing=(
            "hs",
            lambda x: x.isna().sum()
        ),

        hs_imputed=(
            "hs_imputed",
            "sum"
        ),
    )
)

hs_station["recovered_n"] = (
    hs_station["hs_available"]
    -
    hs_station["original_hs"]
)

hs_station["recovery_pct_of_original_missing"] = (
    hs_station["recovered_n"]
    /
    (
        hs_station["rows"]
        -
        hs_station["original_hs"]
    )
    * 100
)

display(
    hs_station.round(3)
)


# ============================================================
# 4. HS FILL METHOD
# ============================================================

print("\n" + "=" * 80)
print("4. HS FILL METHOD")
print("=" * 80)

display(
    pd.crosstab(
        df["station"],
        df["hs_fill_method"]
    )
)


# ============================================================
# 5. ALL IMPUTATION COUNTS
# ============================================================

print("\n" + "=" * 80)
print("5. IMPUTATION COUNTS")
print("=" * 80)

imp_cols = [
    c
    for c in df.columns
    if c.endswith("_imputed")
]

display(
    df[imp_cols]
    .sum()
    .sort_values(ascending=False)
    .to_frame("n_imputed")
)


# ============================================================
# 6. ORIGINAL VS FINAL MISSING
# ============================================================

print("\n" + "=" * 80)
print("6. ORIGINAL VS FINAL")
print("=" * 80)

records = []

for col in BASE_COLS:

    original_flag = (
        f"{col}_original_observed"
    )

    original_missing = (
        len(df)
        -
        df[original_flag].sum()
    )

    final_missing = (
        df[col]
        .isna()
        .sum()
    )

    recovered = (
        original_missing
        -
        final_missing
    )

    records.append({
        "column": col,
        "original_missing":
            int(original_missing),
        "final_missing":
            int(final_missing),
        "recovered":
            int(recovered),
        "recovery_pct":
            (
                recovered
                /
                max(
                    original_missing,
                    1
                )
                * 100
            ),
    })

recovery = pd.DataFrame(records)

display(
    recovery
    .sort_values(
        "recovered",
        ascending=False
    )
    .round(3)
)


# ============================================================
# 7. I-ORS G TRANSFER CHECK
# ============================================================

print("\n" + "=" * 80)
print("7. I-ORS G TRANSFER")
print("=" * 80)

i_transfer = df[
    (df["station"] == "I-ORS")
    &
    (df["hs_fill_method"] == "G_lag_transfer")
]

print(
    "I G-transfer rows:",
    len(i_transfer)
)

if len(i_transfer) > 0:

    print(
        "period:",
        i_transfer["time"].min(),
        "~",
        i_transfer["time"].max()
    )

    print(
        "Hs min/median/max:",
        i_transfer["hs"].min(),
        i_transfer["hs"].median(),
        i_transfer["hs"].max()
    )


# ============================================================
# 8. PHYSICAL SANITY
# ============================================================

print("\n" + "=" * 80)
print("8. PHYSICAL SANITY")
print("=" * 80)

checks = {
    "hs <= 0":
        (
            df["hs"].notna()
            &
            (df["hs"] <= 0)
        ).sum(),

    "tp <= 0":
        (
            df["tp"].notna()
            &
            (df["tp"] <= 0)
        ).sum(),

    "hmax < hs":
        (
            df["hmax"].notna()
            &
            df["hs"].notna()
            &
            (df["hmax"] < df["hs"])
        ).sum(),

    "wspd < 0":
        (
            df["wspd"].notna()
            &
            (df["wspd"] < 0)
        ).sum(),

    "gust < 0":
        (
            df["gust"].notna()
            &
            (df["gust"] < 0)
        ).sum(),

    "relh outside 0~100":
        (
            df["relh"].notna()
            &
            ~df["relh"].between(0, 100)
        ).sum(),

    "caph outside 950~1050":
        (
            df["caph"].notna()
            &
            ~df["caph"].between(950, 1050)
        ).sum(),

    "wdir outside 0~360":
        (
            df["wdir"].notna()
            &
            ~df["wdir"].between(
                0,
                360,
                inclusive="left"
            )
        ).sum(),

    "wvdir outside 0~360":
        (
            df["wvdir"].notna()
            &
            ~df["wvdir"].between(
                0,
                360,
                inclusive="left"
            )
        ).sum(),
}

display(
    pd.Series(
        checks,
        name="violations"
    )
    .to_frame()
)


# ============================================================
# 9. REMAINING HS GAP DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("9. REMAINING HS GAPS")
print("=" * 80)


def get_runs(s):

    missing = (
        s.isna()
        .to_numpy()
    )

    runs = []

    i = 0

    while i < len(s):

        if not missing[i]:
            i += 1
            continue

        start = i

        while (
            i < len(s)
            and missing[i]
        ):
            i += 1

        end = i - 1

        runs.append(
            (
                start,
                end,
                end - start + 1
            )
        )

    return runs


gap_records = []

for station, g in df.groupby("station"):

    g = (
        g.sort_values("time")
        .reset_index(drop=True)
    )

    for start, end, n in get_runs(
        g["hs"]
    ):

        gap_records.append({
            "station": station,
            "start":
                g.loc[start, "time"],
            "end":
                g.loc[end, "time"],
            "steps":
                n,
            "hours":
                n / 6,
            "days":
                n / 144,
        })


hs_gaps = pd.DataFrame(
    gap_records
)

if len(hs_gaps):

    display(
        hs_gaps
        .sort_values(
            "steps",
            ascending=False
        )
        .head(30)
    )

else:

    print(
        "No remaining Hs gaps"
    )


# ============================================================
# 10. MOST IMPORTANT FINAL CHECK
# ============================================================

print("\n" + "=" * 80)
print("10. SUMMARY")
print("=" * 80)

print(
    "Original observed Hs:",
    int(
        df["hs_original_observed"]
        .sum()
    )
)

print(
    "Final available Hs:",
    int(
        df["hs"]
        .notna()
        .sum()
    )
)

print(
    "Newly recovered Hs:",
    int(
        df["hs"]
        .notna()
        .sum()
        -
        df["hs_original_observed"]
        .sum()
    )
)

print(
    "Remaining Hs NaN:",
    int(
        df["hs"]
        .isna()
        .sum()
    )
)

1. BASIC
shape: (183600, 33)
stations: ['G-ORS' 'I-ORS' 'S-ORS']
time: 2024-01-01 00:00:00+09:00 ~ 2025-06-30 23:50:00+09:00

2. FINAL MISSING


,missing_n,missing_pct
relh,69490,37.849
wdir,55281,30.109
airt,55274,30.106
gust,55198,30.064
wspd,55165,30.046
caph,54910,29.907
tp,10529,5.735
hs,10490,5.714
hmax,10485,5.711
wvdir,10303,5.612



3. HS BY STATION


,rows,original_hs,hs_available,hs_missing,hs_imputed,recovered_n,recovery_pct_of_original_missing
station,,,,,,,
G-ORS,78768,38053,76658,2110,38605,38605,94.818
I-ORS,52416,30899,44075,8341,13176,13176,61.235
S-ORS,52416,38978,52377,39,13399,13399,99.710



4. HS FILL METHOD


...


7. I-ORS G TRANSFER
I G-transfer rows: 0

8. PHYSICAL SANITY


,violations
hs <= 0,0
tp <= 0,0
hmax < hs,5
wspd < 0,0
gust < 0,0
relh outside 0~100,0
caph outside 950~1050,0
wdir outside 0~360,0
wvdir outside 0~360,0



9. REMAINING HS GAPS


,station,start,end,steps,hours,days
16,I-ORS,2024-07-24 22:00:00+09:00,2024-11-14 23:40:00+09:00,8142,1357.000000,56.541667
6,G-ORS,2024-10-04 13:50:00+09:00,2024-10-10 16:50:00+09:00,883,147.166667,6.131944
9,G-ORS,2024-11-18 07:30:00+09:00,2024-11-22 15:30:00+09:00,625,104.166667,4.340278
10,G-ORS,2024-11-26 12:10:00+09:00,2024-11-27 14:50:00+09:00,161,26.833333,1.118056
0,G-ORS,2024-04-27 14:50:00+09:00,2024-04-28 10:10:00+09:00,117,19.500000,0.812500
5,G-ORS,2024-09-20 07:30:00+09:00,2024-09-20 23:50:00+09:00,99,16.500000,0.687500
3,G-ORS,2024-08-25 03:10:00+09:00,2024-08-25 18:10:00+09:00,91,15.166667,0.631944
13,I-ORS,2024-07-21 17:20:00+09:00,2024-07-22 09:20:00+09:00,49,8.166667,0.340278
12,I-ORS,2024-07-20 20:00:00+09:00,2024-07-21 11:40:00+09:00,48,8.000000,0.333333
14,I-ORS,2024-07-22 19:00:00+09:00,2024-07-23 09:20:00+09:00,44,7.333333,0.305556



10. SUMMARY
Original observed Hs: 107930
Final available Hs: 173110
Newly recovered Hs: 65180
Remaining Hs NaN: 10490


In [4]:
# ============================================================
# 어떤 변수가 Hs 복원 효과를 막고 있는지 확인
# ============================================================

import pandas as pd
from pathlib import Path

PATH = Path(PROJECT_ROOT / "data" / "processed" / "train_final_v2.csv")

df = pd.read_csv(
    PATH,
    parse_dates=["time"]
)

df = df.sort_values(
    ["station", "time"]
).reset_index(drop=True)


sets = {
    "HS": [
        "hs"
    ],

    "HS_TP": [
        "hs",
        "tp"
    ],

    "HS_TP_HMAX": [
        "hs",
        "tp",
        "hmax"
    ],

    "WAVE_FULL": [
        "hs",
        "tp",
        "hmax",
        "wvdir"
    ],

    "WAVE_WIND": [
        "hs",
        "tp",
        "hmax",
        "wspd",
        "gust"
    ],

    "ALL_BASE": [
        "hs",
        "tp",
        "hmax",
        "wvdir",
        "wspd",
        "gust",
        "wdir",
        "airt",
        "relh",
        "caph"
    ],
}


rows = []

for station, g in df.groupby("station"):

    for name, cols in sets.items():

        complete = (
            g[cols]
            .notna()
            .all(axis=1)
        )

        rows.append({
            "station": station,
            "feature_set": name,
            "complete_rows": int(complete.sum()),
            "complete_pct": complete.mean() * 100,
        })


result = pd.DataFrame(rows)

display(
    result.pivot(
        index="feature_set",
        columns="station",
        values="complete_pct"
    ).round(2)
)


print("\n===== INDIVIDUAL VARIABLE AVAILABILITY =====")

availability = (
    df.groupby("station")[
        [
            "hs",
            "tp",
            "hmax",
            "wvdir",
            "wspd",
            "gust",
            "wdir",
            "airt",
            "relh",
            "caph",
        ]
    ]
    .agg(lambda x: x.notna().mean() * 100)
)

display(
    availability.round(2)
)

station,G-ORS,I-ORS,S-ORS
feature_set,,,
ALL_BASE,93.90,29.36,43.38
HS,97.32,84.09,99.93
HS_TP,97.23,84.08,99.93
HS_TP_HMAX,97.23,84.08,99.93
WAVE_FULL,96.90,84.08,99.84
WAVE_WIND,95.35,49.67,49.69



===== INDIVIDUAL VARIABLE AVAILABILITY =====


,hs,tp,hmax,wvdir,wspd,gust,wdir,airt,relh,caph
station,,,,,,,,,,
G-ORS,97.32,97.27,97.33,97.59,96.88,96.89,96.85,97.12,96.34,97.20
I-ORS,84.09,84.08,84.09,84.13,49.73,49.68,49.57,49.15,29.43,49.73
S-ORS,99.93,99.93,99.93,99.84,49.72,49.70,49.70,49.73,43.50,49.73


In [5]:
# ============================================================
# NEXT CELL
# CREATE train_final_physics_v2.csv
#
# INPUT
#   train_final_v2.csv
#
# OUTPUT
#   train_final_physics_v2.csv
#
# 중요:
# - base 값 수정 X
# - 추가 결측 복원 X
# - physics/derived feature만 추가
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# PATH
# ============================================================

PATH = Path(PROJECT_ROOT / "data" / "processed" / "train_final_v2.csv")
PHYSICS_PATH = Path(PROJECT_ROOT / "data" / "processed" / "train_final_physics_v2.csv")


# ============================================================
# LOAD
# ============================================================

df = pd.read_csv(
    PATH,
    parse_dates=["time"]
)

df = (
    df
    .sort_values(["station", "time"])
    .reset_index(drop=True)
)

print("INPUT SHAPE:", df.shape)


# ============================================================
# CONSTANTS
# ============================================================

G = 9.80665
EPS = 1e-6


# ============================================================
# 1. DIRECTION CYCLIC
# ============================================================

wdir_rad = np.deg2rad(df["wdir"])
wvdir_rad = np.deg2rad(df["wvdir"])

df["wdir_sin"] = np.sin(wdir_rad)
df["wdir_cos"] = np.cos(wdir_rad)

df["wvdir_sin"] = np.sin(wvdir_rad)
df["wvdir_cos"] = np.cos(wvdir_rad)


# ============================================================
# 2. WIND VECTOR
# ============================================================

df["u_wind"] = (
    df["wspd"]
    *
    np.sin(wdir_rad)
)

df["v_wind"] = (
    df["wspd"]
    *
    np.cos(wdir_rad)
)


# ============================================================
# 3. WIND FORCE
# ============================================================

df["wspd2"] = df["wspd"] ** 2
df["wspd3"] = df["wspd"] ** 3

df["gust_minus_wspd"] = (
    df["gust"]
    -
    df["wspd"]
)


# ============================================================
# 4. WIND ↔ WAVE DIRECTION
# ============================================================

direction_diff = (
    (
        df["wdir"]
        -
        df["wvdir"]
        +
        180
    )
    %
    360
    -
    180
)

df["wind_wave_diff"] = (
    np.abs(direction_diff)
)

df["wind_wave_alignment"] = (
    np.cos(
        np.deg2rad(direction_diff)
    )
)


# ============================================================
# GROUP BY STATION
# ============================================================

groups = df.groupby(
    "station",
    group_keys=False
)


# ============================================================
# 5. WIND HISTORY
# 10-min grid
# ============================================================

df["wspd_mean_3h"] = (
    groups["wspd"]
    .transform(
        lambda s:
        s.rolling(
            18,
            min_periods=6
        ).mean()
    )
)

df["wspd_mean_6h"] = (
    groups["wspd"]
    .transform(
        lambda s:
        s.rolling(
            36,
            min_periods=12
        ).mean()
    )
)

df["wspd_mean_12h"] = (
    groups["wspd"]
    .transform(
        lambda s:
        s.rolling(
            72,
            min_periods=24
        ).mean()
    )
)


df["gust_max_3h"] = (
    groups["gust"]
    .transform(
        lambda s:
        s.rolling(
            18,
            min_periods=6
        ).max()
    )
)

df["gust_max_6h"] = (
    groups["gust"]
    .transform(
        lambda s:
        s.rolling(
            36,
            min_periods=12
        ).max()
    )
)

df["gust_max_12h"] = (
    groups["gust"]
    .transform(
        lambda s:
        s.rolling(
            72,
            min_periods=24
        ).max()
    )
)


df["wspd3_mean_6h"] = (
    groups["wspd"]
    .transform(
        lambda s:
        (s ** 3)
        .rolling(
            36,
            min_periods=12
        )
        .mean()
    )
)


# ============================================================
# 6. PRESSURE TENDENCY
# ============================================================

df["caph_change_3h"] = (
    df["caph"]
    -
    groups["caph"].shift(18)
)

df["caph_change_6h"] = (
    df["caph"]
    -
    groups["caph"].shift(36)
)

df["caph_change_12h"] = (
    df["caph"]
    -
    groups["caph"].shift(72)
)


# ============================================================
# 7. PURE WAVE PHYSICS
# ============================================================

# Hs^2
df["wave_energy_proxy"] = (
    df["hs"] ** 2
)


# Hs^2 * Tp
df["wave_power_proxy"] = (
    df["hs"] ** 2
    *
    df["tp"]
)


# Hs / Tp^2
df["wave_steepness_proxy"] = (
    df["hs"]
    /
    (
        df["tp"] ** 2
        +
        EPS
    )
)


# Hs / Tp
df["hs_tp_ratio"] = (
    df["hs"]
    /
    (
        df["tp"]
        +
        EPS
    )
)


# ============================================================
# 8. DEEP-WATER WAVE PHYSICS
# ============================================================

# L = g T^2 / 2pi
df["deepwater_wavelength"] = (
    G
    *
    df["tp"] ** 2
    /
    (
        2
        *
        np.pi
    )
)


# Cp = gT / 2pi
df["phase_speed_proxy"] = (
    G
    *
    df["tp"]
    /
    (
        2
        *
        np.pi
    )
)


# H / L
df["physical_steepness"] = (
    df["hs"]
    /
    (
        df["deepwater_wavelength"]
        +
        EPS
    )
)


# ============================================================
# 9. WIND-WAVE COUPLING
#
# wspd가 필요하기 때문에
# PURE_WAVE_PHYSICS와 따로 ablation 할 것
# ============================================================

df["wave_age_proxy"] = (
    df["phase_speed_proxy"]
    /
    (
        df["wspd"]
        +
        1.0
    )
)


# ============================================================
# 10. TIME FEATURES
# ============================================================

hour = (
    df["time"].dt.hour
    +
    df["time"].dt.minute / 60.0
)

month = df["time"].dt.month


df["hour_sin"] = np.sin(
    2 * np.pi * hour / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * hour / 24
)


df["month_sin"] = np.sin(
    2 * np.pi * (month - 1) / 12
)

df["month_cos"] = np.cos(
    2 * np.pi * (month - 1) / 12
)


# ============================================================
# 11. INF -> NaN
# ============================================================

numeric_cols = (
    df
    .select_dtypes(include=np.number)
    .columns
)

df[numeric_cols] = (
    df[numeric_cols]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)


# ============================================================
# 12. SAVE
# ============================================================

df.to_csv(
    PHYSICS_PATH,
    index=False
)


# ============================================================
# 13. CHECK
# ============================================================

physics_features = [
    # pure wave / low missingness
    "wave_energy_proxy",
    "wave_power_proxy",
    "wave_steepness_proxy",
    "hs_tp_ratio",
    "deepwater_wavelength",
    "phase_speed_proxy",
    "physical_steepness",

    # wave direction
    "wvdir_sin",
    "wvdir_cos",

    # wind
    "u_wind",
    "v_wind",
    "wspd2",
    "wspd3",
    "gust_minus_wspd",
    "wspd_mean_3h",
    "wspd_mean_6h",
    "wspd_mean_12h",
    "gust_max_3h",
    "gust_max_6h",
    "gust_max_12h",
    "wspd3_mean_6h",

    # pressure
    "caph_change_3h",
    "caph_change_6h",
    "caph_change_12h",

    # direction coupling
    "wdir_sin",
    "wdir_cos",
    "wind_wave_diff",
    "wind_wave_alignment",

    # wind-wave
    "wave_age_proxy",

    # time
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
]


print("\n" + "=" * 80)
print("PHYSICS V2 CREATED")
print("=" * 80)

print("Saved:", PHYSICS_PATH)
print("Shape:", df.shape)

print("\n===== BASE HS MUST NOT CHANGE =====")

print(
    "Hs available:",
    int(df["hs"].notna().sum())
)

print(
    "Hs missing:",
    int(df["hs"].isna().sum())
)

print(
    "Original observed Hs:",
    int(df["hs_original_observed"].sum())
)


print("\n===== PHYSICS FEATURE AVAILABILITY =====")

availability = pd.DataFrame({
    "available_n":
        df[physics_features]
        .notna()
        .sum(),

    "available_pct":
        df[physics_features]
        .notna()
        .mean()
        * 100,

    "missing_n":
        df[physics_features]
        .isna()
        .sum(),

    "missing_pct":
        df[physics_features]
        .isna()
        .mean()
        * 100,
})

display(
    availability
    .sort_values(
        "available_pct",
        ascending=False
    )
    .round(2)
)

INPUT SHAPE: (183600, 33)

PHYSICS V2 CREATED
Saved: train_final_physics_v2.csv
Shape: (183600, 66)

===== BASE HS MUST NOT CHANGE =====
Hs available: 173110
Hs missing: 10490
Original observed Hs: 107930

===== PHYSICS FEATURE AVAILABILITY =====


,available_n,available_pct,missing_n,missing_pct
month_cos,183600,100.00,0,0.00
month_sin,183600,100.00,0,0.00
hour_cos,183600,100.00,0,0.00
hour_sin,183600,100.00,0,0.00
wvdir_sin,173297,94.39,10303,5.61
wvdir_cos,173297,94.39,10303,5.61
wave_energy_proxy,173110,94.29,10490,5.71
deepwater_wavelength,173071,94.27,10529,5.73
phase_speed_proxy,173071,94.27,10529,5.73
physical_steepness,173040,94.25,10560,5.75


In [2]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# EXP04 Full Reproduction Pipeline (From Raw Data to Submission)\n",
        "\n",
        "대회 원본 관측 데이터(`train_atmos.csv`, `train_wave.csv`)로부터 시작하여, 전처리 및 물리 피처 엔지니어링, EXP04 최종 하이퍼파라미터 모델 학습, 테스트 추론 및 제출 파일 생성까지 한 번에 수행하는 전체 재현 노트북입니다.\n",
        "\n",
        "### 파이프라인 단계\n",
        "1. **Raw Merge & Grid Alignment (`train_v2`)**: 해양/기상 원본 결합 및 3개 기지 10분 정규 격자(236,304행) 구축\n",
        "2. **Physics Imputation (`train_v4` Stage 1 & 2)**: 1스텝 보간, Rayleigh 파고 역학, Wilson 풍파 평형, 기압/온습도 미기후 물리 대치\n",
        "3. **Physics Feature Generation (`train_v4` Stage 3)**: EXP04 최적 31개 피처(`ALL_FEATURES`) 산출\n",
        "4. **iTransformer Final Model Training**: Optuna Trial 25 최적 하이퍼파라미터 및 `CompetitionAlignedRMSELoss` 기반 학습\n",
        "5. **Test Inference & Submission**: `test_context.parquet` 결측 방어 처리 후 6개 리드타임 예측 및 `submission_exp04.csv` 생성"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# ============================================================\n",
        "# 0. SETUP & ENVIRONMENT\n",
        "# ============================================================\n",
        "from pathlib import Path\n",
        "import gc\n",
        "import json\n",
        "import math\n",
        "import random\n",
        "import warnings\n",
        "\n",
        "import numpy as np\n",
        "import pandas as pd\n",
        "import matplotlib.pyplot as plt\n",
        "\n",
        "from sklearn.preprocessing import StandardScaler\n",
        "from sklearn.metrics import mean_squared_error\n",
        "\n",
        "import torch\n",
        "import torch.nn as nn\n",
        "from torch.utils.data import Dataset, DataLoader\n",
        "\n",
        "warnings.filterwarnings(\"ignore\")\n",
        "\n",
        "SEED = 42\n",
        "\n",
        "def seed_everything(seed=SEED):\n",
        "    random.seed(seed)\n",
        "    np.random.seed(seed)\n",
        "    torch.manual_seed(seed)\n",
        "    torch.cuda.manual_seed_all(seed)\n",
        "\n",
        "seed_everything()\n",
        "\n",
        "DEVICE = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n",
        "print(\"DEVICE:\", DEVICE)\n",
        "if torch.cuda.is_available():\n",
        "    print(\"GPU:\", torch.cuda.get_device_name(0))\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# ============================================================\n",
        "# 1. CONFIG & EXP04 BEST HYPERPARAMETERS\n",
        "# ============================================================\n",
        "RAW_ATMOS_PATH = Path(\"train_atmos.csv\")\n",
        "RAW_WAVE_PATH = Path(\"train_wave.csv\")\n",
        "TEST_CONTEXT_PATH = Path(\"test_context.parquet\")\n",
        "TEST_INDEX_PATH = Path(\"test_index.csv\")\n",
        "SUBMISSION_PATH = Path(\"submission_exp04.csv\")\n",
        "\n",
        "EXP_DIR = Path(\"experiments/exp04_full_reproduce\")\n",
        "EXP_DIR.mkdir(parents=True, exist_ok=True)\n",
        "\n",
        "TIME_COL = \"time\"\n",
        "STATION_COL = \"station\"\n",
        "STEP_MINUTES = 10\n",
        "STEPS_PER_HOUR = 6\n",
        "\n",
        "INPUT_LEN = 289\n",
        "LEAD_HOURS = [3, 6, 9, 12, 18, 24]\n",
        "LEAD_STEPS = [h * STEPS_PER_HOUR for h in LEAD_HOURS]\n",
        "MAX_LEAD = max(LEAD_STEPS)\n",
        "N_TARGETS = len(LEAD_STEPS)\n",
        "\n",
        "TRAIN_RATIO = 0.80\n",
        "FINAL_EPOCHS = 35\n",
        "FINAL_PATIENCE = 6\n",
        "\n",
        "# EXP04 최종 확정 피처셋 (31개 ALL_FEATURES)\n",
        "BEST_FEATURES = [\n",
        "    'hs', 'tp', 'hmax', 'wspd', 'gust', 'u_wind', 'v_wind', 'hs_diff_1h', 'hs_diff_3h',\n",
        "    'hs_mean_6h', 'hs_mean_12h', 'hs_max_6h', 'hs_max_12h', 'wave_steepness', 'wave_energy',\n",
        "    'effective_wind_forcing', 'u_wave', 'v_wave', 'wspd_mean_6h', 'wspd_mean_12h',\n",
        "    'gust_max_6h', 'gust_max_12h', 'gust_minus_wspd', 'caph_change_3h', 'caph_change_6h',\n",
        "    'caph_change_12h', 'wind_wave_alignment', 'wind_wave_diff', 'airt', 'relh', 'caph'\n",
        "]\n",
        "\n",
        "# EXP04 Optuna Trial 25 최종 최적 하이퍼파라미터\n",
        "FINAL_PARAMS = {\n",
        "    \"d_model\": 64,\n",
        "    \"n_heads\": 8,\n",
        "    \"e_layers\": 4,\n",
        "    \"dropout\": 0.3460508306239601,\n",
        "    \"lr\": 9.11924776137983e-06,\n",
        "    \"weight_decay\": 4.815394773868456e-06,\n",
        "    \"batch_size\": 64,\n",
        "    \"d_ff\": 64 * 4,\n",
        "}\n",
        "\n",
        "print(\"Best Features Count:\", len(BEST_FEATURES))\n",
        "print(\"Final Model Hyperparameters:\", FINAL_PARAMS)\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# ============================================================\n",
        "# 2. STEP 1: RAW MERGE & REGULAR 10-MIN GRID ALIGNMENT\n",
        "# ============================================================\n",
        "print(\"\\n\" + \"=\" * 70)\n",
        "print(\"[Step 1/5] Merging raw files & constructing regular 10-min grid...\")\n",
        "print(\"=\" * 70)\n",
        "\n",
        "wave = pd.read_csv(RAW_WAVE_PATH, parse_dates=[TIME_COL])\n",
        "atmos = pd.read_csv(RAW_ATMOS_PATH, parse_dates=[TIME_COL])\n",
        "\n",
        "raw_df = pd.merge(wave, atmos, on=[STATION_COL, TIME_COL], how=\"outer\", validate=\"one_to_one\")\n",
        "base_cols = [\"station\", \"time\", \"hs\", \"tp\", \"hmax\", \"wvdir\", \"wspd\", \"gust\", \"wdir\", \"airt\", \"relh\", \"caph\"]\n",
        "raw_df = raw_df[[c for c in base_cols if c in raw_df.columns]].sort_values([STATION_COL, TIME_COL]).reset_index(drop=True)\n",
        "\n",
        "min_time = raw_df[TIME_COL].min()\n",
        "max_time = raw_df[TIME_COL].max()\n",
        "full_grid = pd.date_range(min_time, max_time, freq=\"10min\", tz=raw_df[TIME_COL].dt.tz)\n",
        "stations = sorted(raw_df[STATION_COL].dropna().unique())\n",
        "\n",
        "grid_frames = []\n",
        "for st in stations:\n",
        "    st_frame = raw_df[raw_df[STATION_COL] == st].drop_duplicates(subset=[TIME_COL])\n",
        "    st_grid = pd.DataFrame({STATION_COL: st, TIME_COL: full_grid})\n",
        "    merged_st = pd.merge(st_grid, st_frame, on=[STATION_COL, TIME_COL], how=\"left\")\n",
        "    grid_frames.append(merged_st)\n",
        "\n",
        "train_v2 = pd.concat(grid_frames, ignore_index=True).sort_values([STATION_COL, TIME_COL]).reset_index(drop=True)\n",
        "train_v2[\"hs_original_observed\"] = train_v2[\"hs\"].notna().astype(np.int8)\n",
        "print(f\"-> train_v2 shape: {train_v2.shape} (78,768 rows x {len(stations)} stations = 236,304 rows)\")\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# ============================================================\n",
        "# 3. STEP 2: PHYSICS-BASED IMPUTATION\n",
        "# ============================================================\n",
        "print(\"\\n\" + \"=\" * 70)\n",
        "print(\"[Step 2/5] Running physics imputation (minimal + station equilibrium)...\")\n",
        "print(\"=\" * 70)\n",
        "\n",
        "# [Stage 1] 1스텝 보간 및 동시간대 상호 물리 대치\n",
        "def process_station_minimal(group):\n",
        "    g = group.sort_values(TIME_COL).set_index(TIME_COL).copy()\n",
        "    st = g[STATION_COL].iloc[0]\n",
        "    \n",
        "    rayleigh = 1.64 if st == 'I-ORS' else 1.62 if st == 'G-ORS' else 1.63\n",
        "    gust_f = 1.10 if st == 'I-ORS' else 1.12 if st == 'G-ORS' else 1.15\n",
        "    tp_f = 4.10 if st == 'I-ORS' else 3.75 if st == 'G-ORS' else 3.60\n",
        "    sensor_h = 41.0 if st == 'I-ORS' else 31.0 if st == 'G-ORS' else 30.0\n",
        "    alpha = 0.11 if st == 'I-ORS' else 0.13 if st == 'G-ORS' else 0.14\n",
        "    h_to_u10 = (10.0 / sensor_h) ** alpha\n",
        "\n",
        "    num_cols = ['hs', 'tp', 'hmax', 'wspd', 'gust', 'airt', 'relh', 'caph']\n",
        "    g[num_cols] = g[num_cols].interpolate(method='time', limit=1)\n",
        "\n",
        "    for col in ['wdir', 'wvdir']:\n",
        "        if col in g.columns and g[col].notna().any():\n",
        "            rad = np.deg2rad(g[col])\n",
        "            u = np.cos(rad).interpolate(method='time', limit=1)\n",
        "            v = np.sin(rad).interpolate(method='time', limit=1)\n",
        "            g[col] = (np.rad2deg(np.arctan2(v, u)) % 360.0).round(2)\n",
        "\n",
        "    g['wdir'] = g['wdir'].fillna(g['wvdir'])\n",
        "    g['wvdir'] = g['wvdir'].fillna(g['wdir'])\n",
        "\n",
        "    g['hs'] = g['hs'].fillna(g['hmax'] / rayleigh)\n",
        "    g['hmax'] = g['hmax'].fillna(g['hs'] * rayleigh)\n",
        "\n",
        "    estimated_u10 = np.sqrt(g['hs'].dropna().clip(lower=0) / 0.0246)\n",
        "    estimated_wspd = estimated_u10 / h_to_u10\n",
        "    g['wspd'] = g['wspd'].fillna(g['gust'] / gust_f)\n",
        "    g['wspd'] = g['wspd'].fillna(estimated_wspd)\n",
        "    g['gust'] = g['gust'].fillna(g['wspd'] * gust_f)\n",
        "\n",
        "    u10 = g['wspd'] * h_to_u10\n",
        "    g['hs'] = g['hs'].fillna(0.0246 * (u10 ** 2))\n",
        "    g['hmax'] = g['hmax'].fillna(g['hs'] * rayleigh)\n",
        "    g['tp'] = g['tp'].fillna(tp_f * np.sqrt(g['hs'].clip(lower=0.01)))\n",
        "    return g.reset_index()\n",
        "\n",
        "stage1_stations = [process_station_minimal(group) for _, group in train_v2.groupby(STATION_COL, sort=False)]\n",
        "df_stage1 = pd.concat(stage1_stations, ignore_index=True)\n",
        "\n",
        "# [Stage 2] 관측소별 미기후 주기 및 풍파 에너지 평형 대치\n",
        "class StationPhysicsImputer:\n",
        "    def __init__(self):\n",
        "        self.STATION_SPECS = {\n",
        "            'S-ORS': {'temp_amp': 2.4, 'mean_p': 1015.0, 'gust_factor': 1.15, 'rayleigh': 1.63, 'tp_factor': 3.60},\n",
        "            'G-ORS': {'temp_amp': 1.6, 'mean_p': 1013.5, 'gust_factor': 1.12, 'rayleigh': 1.62, 'tp_factor': 3.75},\n",
        "            'I-ORS': {'temp_amp': 1.0, 'mean_p': 1011.8, 'gust_factor': 1.10, 'rayleigh': 1.64, 'tp_factor': 4.10}\n",
        "        }\n",
        "\n",
        "    def process_station(self, group: pd.DataFrame, st_name: str) -> pd.DataFrame:\n",
        "        g = group.sort_values(TIME_COL).copy()\n",
        "        specs = self.STATION_SPECS.get(st_name, self.STATION_SPECS['G-ORS'])\n",
        "        month = g[TIME_COL].dt.month\n",
        "        hour = g[TIME_COL].dt.hour + g[TIME_COL].dt.minute / 60.0\n",
        "\n",
        "        seasonal_airt = g.groupby(month)['airt'].transform('mean')\n",
        "        diurnal_cycle = specs['temp_amp'] * np.sin(2 * np.pi * (hour - 8.0) / 24.0)\n",
        "        g['airt'] = g['airt'].fillna(seasonal_airt + diurnal_cycle).bfill().ffill()\n",
        "\n",
        "        seasonal_relh = g.groupby(month)['relh'].transform('mean')\n",
        "        g['relh'] = g['relh'].fillna(seasonal_relh - (diurnal_cycle * 3.5)).clip(15.0, 100.0).bfill().ffill()\n",
        "\n",
        "        seasonal_caph = g.groupby(month)['caph'].transform('mean')\n",
        "        g['caph'] = g['caph'].fillna(seasonal_caph).fillna(specs['mean_p']).bfill().ffill()\n",
        "\n",
        "        winter_mask = month.isin([11, 12, 1, 2])\n",
        "        default_wspd = pd.Series(np.where(winter_mask, 8.5, 6.0), index=g.index)\n",
        "        default_wdir = pd.Series(np.where(winter_mask, 315.0, 180.0), index=g.index)\n",
        "\n",
        "        g['wspd'] = g['wspd'].fillna(default_wspd).bfill().ffill()\n",
        "        g['gust'] = g['gust'].fillna(g['wspd'] * specs['gust_factor']).bfill().ffill()\n",
        "        g['wdir'] = g['wdir'].fillna(default_wdir).bfill().ffill()\n",
        "        g['wvdir'] = g['wvdir'].fillna(g['wdir']).bfill().ffill()\n",
        "\n",
        "        u10 = g['wspd'] * 0.86\n",
        "        smb_hs = (0.243 * (u10 ** 2) / 9.81).clip(lower=0.2)\n",
        "        g['hs'] = g['hs'].fillna(smb_hs).bfill().ffill()\n",
        "        g['hmax'] = g['hmax'].fillna(g['hs'] * specs['rayleigh']).bfill().ffill()\n",
        "        g['tp'] = g['tp'].fillna(specs['tp_factor'] * np.sqrt(g['hs'].clip(lower=0.01))).bfill().ffill()\n",
        "\n",
        "        g['wspd'] = g['wspd'].clip(0, 45.0)\n",
        "        g['gust'] = np.maximum(g['gust'].clip(0, 60.0), g['wspd'])\n",
        "        g['hs'] = g['hs'].clip(0.01, 15.0)\n",
        "        g['hmax'] = np.maximum(g['hmax'].clip(0.01, 20.0), g['hs'])\n",
        "        g['tp'] = g['tp'].clip(1.0, 25.0)\n",
        "        g['wdir'] = g['wdir'] % 360.0\n",
        "        g['wvdir'] = g['wvdir'] % 360.0\n",
        "        return g\n",
        "\n",
        "    def run(self, full_df: pd.DataFrame) -> pd.DataFrame:\n",
        "        results = []\n",
        "        for st_name, group in full_df.groupby(STATION_COL):\n",
        "            results.append(self.process_station(group, st_name))\n",
        "        return pd.concat(results, axis=0).sort_values([STATION_COL, TIME_COL]).reset_index(drop=True)\n",
        "\n",
        "imputer = StationPhysicsImputer()\n",
        "df_stage2 = imputer.run(df_stage1)\n",
        "print(\"-> Physics Imputation Completed. Remaining NaNs:\", int(df_stage2[base_cols].isna().sum().sum()))\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# ============================================================\n",
        "# 4. STEP 3: PHYSICS FEATURE GENERATION (train_final_physics_v4)\n",
        "# ============================================================\n",
        "print(\"\\n\" + \"=\" * 70)\n",
        "print(\"[Step 3/5] Computing expanded physics features (31 ALL_FEATURES)...\")\n",
        "print(\"=\" * 70)\n",
        "\n",
        "def add_physics_features(group: pd.DataFrame) -> pd.DataFrame:\n",
        "    g = group.sort_values(TIME_COL).copy()\n",
        "    g[\"wspd_mean_6h\"] = g[\"wspd\"].rolling(36, min_periods=12).mean()\n",
        "    g[\"wspd_mean_12h\"] = g[\"wspd\"].rolling(72, min_periods=24).mean()\n",
        "    g[\"gust_max_6h\"] = g[\"gust\"].rolling(36, min_periods=12).max()\n",
        "    g[\"gust_max_12h\"] = g[\"gust\"].rolling(72, min_periods=24).max()\n",
        "    g[\"gust_minus_wspd\"] = g[\"gust\"] - g[\"wspd\"]\n",
        "\n",
        "    angle_diff = np.deg2rad(g[\"wdir\"] - g[\"wvdir\"])\n",
        "    g[\"wind_wave_alignment\"] = np.cos(angle_diff)\n",
        "    g[\"wind_wave_diff\"] = np.abs((g[\"wdir\"] - g[\"wvdir\"] + 180) % 360 - 180)\n",
        "\n",
        "    g[\"caph_change_3h\"] = g[\"caph\"] - g[\"caph\"].shift(18)\n",
        "    g[\"caph_change_6h\"] = g[\"caph\"] - g[\"caph\"].shift(36)\n",
        "    g[\"caph_change_12h\"] = g[\"caph\"] - g[\"caph\"].shift(72)\n",
        "\n",
        "    g[\"hs_diff_1h\"] = g[\"hs\"] - g[\"hs\"].shift(6)\n",
        "    g[\"hs_diff_3h\"] = g[\"hs\"] - g[\"hs\"].shift(18)\n",
        "    g[\"hs_mean_6h\"] = g[\"hs\"].rolling(36, min_periods=12).mean()\n",
        "    g[\"hs_mean_12h\"] = g[\"hs\"].rolling(72, min_periods=24).mean()\n",
        "    g[\"hs_max_6h\"] = g[\"hs\"].rolling(36, min_periods=12).max()\n",
        "    g[\"hs_max_12h\"] = g[\"hs\"].rolling(72, min_periods=24).max()\n",
        "\n",
        "    wavelength = 1.56 * (g[\"tp\"] ** 2)\n",
        "    g[\"wave_steepness\"] = g[\"hs\"] / np.maximum(wavelength, 1.0)\n",
        "    g[\"wave_energy\"] = g[\"hs\"] ** 2\n",
        "    g[\"effective_wind_forcing\"] = (g[\"wspd\"] ** 2) * g[\"wind_wave_alignment\"]\n",
        "\n",
        "    wave_radians = np.deg2rad(g[\"wvdir\"])\n",
        "    g[\"u_wave\"] = g[\"hs\"] * np.sin(wave_radians)\n",
        "    g[\"v_wave\"] = g[\"hs\"] * np.cos(wave_radians)\n",
        "    return g\n",
        "\n",
        "physics_list = [add_physics_features(group) for _, group in df_stage2.groupby(STATION_COL)]\n",
        "train_final_physics = pd.concat(physics_list, ignore_index=True).sort_values([STATION_COL, TIME_COL]).reset_index(drop=True)\n",
        "\n",
        "wind_radians = np.deg2rad(train_final_physics[\"wdir\"])\n",
        "train_final_physics[\"u_wind\"] = train_final_physics[\"wspd\"] * np.sin(wind_radians)\n",
        "train_final_physics[\"v_wind\"] = train_final_physics[\"wspd\"] * np.cos(wind_radians)\n",
        "\n",
        "# 롤링 초기 경계 결측치 보정 (동일 관측소 내 bfill/ffill)\n",
        "feat_cols = [c for c in train_final_physics.columns if c not in [STATION_COL, TIME_COL, \"hs_original_observed\"]]\n",
        "train_final_physics[feat_cols] = train_final_physics.groupby(STATION_COL)[feat_cols].transform(lambda g: g.bfill().ffill())\n",
        "\n",
        "train_final_physics.to_csv(\"train_final_physics_v4.csv\", index=False)\n",
        "print(f\"-> train_final_physics_v4.csv saved! Shape: {train_final_physics.shape}\")\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# ============================================================\n",
        "# 5. STEP 4: EXP04 iTransformer MODEL TRAINING\n",
        "# ============================================================\n",
        "print(\"\\n\" + \"=\" * 70)\n",
        "print(\"[Step 4/5] Training EXP04 iTransformer model...\")\n",
        "print(\"=\" * 70)\n",
        "\n",
        "STATION_TO_ID = {station: i for i, station in enumerate(sorted(train_final_physics[STATION_COL].unique()))}\n",
        "WINDOW_META_KEYS = (\"station_id\", \"start_idx\", \"end_idx\", \"origin_hs\", \"origin_time\")\n",
        "\n",
        "def build_samples(frame, features, input_len=INPUT_LEN, lead_steps=LEAD_STEPS):\n",
        "    station_id, start_idx, end_idx = [], [], []\n",
        "    origin_hs, origin_time = [], []\n",
        "    x_by_station, hs_by_station = {}, {}\n",
        "    max_lead = max(lead_steps)\n",
        "\n",
        "    for station, g in frame.groupby(STATION_COL, sort=False):\n",
        "        g = g.sort_values(TIME_COL).reset_index(drop=True)\n",
        "        sid = STATION_TO_ID[station]\n",
        "        Xv = g.loc[:, features].to_numpy(dtype=np.float32, copy=True)\n",
        "        hsv = g[\"hs\"].to_numpy(dtype=np.float32, copy=True)\n",
        "        obs = g[\"hs_original_observed\"].to_numpy(dtype=np.int8, copy=True)\n",
        "        times = g[TIME_COL].to_numpy(copy=True)\n",
        "        x_by_station[sid] = Xv\n",
        "        hs_by_station[sid] = hsv\n",
        "        dt = pd.Series(g[TIME_COL]).diff().dt.total_seconds().div(60).to_numpy()\n",
        "\n",
        "        for e in range(input_len - 1, len(g) - max_lead):\n",
        "            s = e - input_len + 1\n",
        "            if not np.all(dt[s + 1:e + 1] == STEP_MINUTES):\n",
        "                continue\n",
        "            if not np.isfinite(Xv[s:e + 1]).all():\n",
        "                continue\n",
        "            target_idx = np.asarray([e + step for step in lead_steps], dtype=np.int64)\n",
        "            if not np.isfinite(hsv[target_idx]).all() or not np.all(obs[target_idx] == 1):\n",
        "                continue\n",
        "            if not np.isfinite(hsv[e]):\n",
        "                continue\n",
        "\n",
        "            station_id.append(sid)\n",
        "            start_idx.append(s)\n",
        "            end_idx.append(e)\n",
        "            origin_hs.append(hsv[e])\n",
        "            origin_time.append(times[e])\n",
        "\n",
        "    return {\n",
        "        \"station_id\": np.asarray(station_id, dtype=np.int64),\n",
        "        \"start_idx\": np.asarray(start_idx, dtype=np.int64),\n",
        "        \"end_idx\": np.asarray(end_idx, dtype=np.int64),\n",
        "        \"origin_hs\": np.asarray(origin_hs, dtype=np.float32),\n",
        "        \"origin_time\": np.asarray(origin_time, dtype=\"datetime64[ns]\"),\n",
        "        \"X_by_station\": x_by_station,\n",
        "        \"hs_by_station\": hs_by_station,\n",
        "        \"source_frame\": frame,\n",
        "        \"features\": list(features),\n",
        "        \"n_features\": len(features),\n",
        "    }\n",
        "\n",
        "def chronological_split(samples, train_ratio=TRAIN_RATIO):\n",
        "    times = pd.to_datetime(samples[\"origin_time\"])\n",
        "    cutoff = pd.Timestamp(pd.Series(times).quantile(train_ratio))\n",
        "\n",
        "    def subset(mask):\n",
        "        part = {key: samples[key][mask] for key in WINDOW_META_KEYS}\n",
        "        part.update({\n",
        "            \"X_by_station\": samples[\"X_by_station\"],\n",
        "            \"hs_by_station\": samples[\"hs_by_station\"],\n",
        "            \"source_frame\": samples[\"source_frame\"],\n",
        "            \"features\": samples[\"features\"],\n",
        "            \"n_features\": samples[\"n_features\"],\n",
        "            \"split_cutoff\": cutoff,\n",
        "        })\n",
        "        return part\n",
        "\n",
        "    return subset(times <= cutoff), subset(times > cutoff), cutoff\n",
        "\n",
        "def scale_samples(train, valid):\n",
        "    scaler = StandardScaler()\n",
        "    train_rows = train[\"source_frame\"][TIME_COL] <= train[\"split_cutoff\"]\n",
        "    scaler.fit(train[\"source_frame\"].loc[train_rows, train[\"features\"]])\n",
        "    for Xv in train[\"X_by_station\"].values():\n",
        "        scaler.transform(Xv, copy=False)\n",
        "    return dict(train), dict(valid), scaler\n",
        "\n",
        "samples = build_samples(train_final_physics, BEST_FEATURES)\n",
        "train_samples, valid_samples, split_cutoff = chronological_split(samples)\n",
        "train_samples, valid_samples, scaler = scale_samples(train_samples, valid_samples)\n",
        "\n",
        "class WaveDataset(Dataset):\n",
        "    def __init__(self, samples):\n",
        "        self.X_by_station = samples[\"X_by_station\"]\n",
        "        self.hs_by_station = samples[\"hs_by_station\"]\n",
        "        self.station_id = samples[\"station_id\"]\n",
        "        self.start_idx = samples[\"start_idx\"]\n",
        "        self.end_idx = samples[\"end_idx\"]\n",
        "        self.origin_hs = samples[\"origin_hs\"]\n",
        "        self.lead_steps = np.asarray(LEAD_STEPS, dtype=np.int64)\n",
        "\n",
        "    def __len__(self):\n",
        "        return len(self.end_idx)\n",
        "\n",
        "    def __getitem__(self, idx):\n",
        "        sid = int(self.station_id[idx])\n",
        "        start, end = int(self.start_idx[idx]), int(self.end_idx[idx])\n",
        "        X = torch.from_numpy(self.X_by_station[sid][start:end + 1])\n",
        "        y = torch.from_numpy(self.hs_by_station[sid][end + self.lead_steps])\n",
        "        return X, y, torch.tensor(self.origin_hs[idx], dtype=torch.float32), torch.tensor(sid)\n",
        "\n",
        "def make_loader(samples, batch_size=64, shuffle=False):\n",
        "    return DataLoader(WaveDataset(samples), batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=False)\n",
        "\n",
        "class ITransformer(nn.Module):\n",
        "    def __init__(self, seq_len, n_features, pred_len, d_model=64, n_heads=8, e_layers=4, dropout=0.346, d_ff=None):\n",
        "        super().__init__()\n",
        "        self.seq_len = seq_len\n",
        "        self.n_features = n_features\n",
        "        self.pred_len = pred_len\n",
        "        if d_ff is None:\n",
        "            d_ff = d_model * 4\n",
        "\n",
        "        self.value_embedding = nn.Linear(seq_len, d_model)\n",
        "        encoder_layer = nn.TransformerEncoderLayer(\n",
        "            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff, dropout=dropout,\n",
        "            batch_first=True, activation=\"gelu\", norm_first=True\n",
        "        )\n",
        "        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=e_layers)\n",
        "        self.norm = nn.LayerNorm(d_model)\n",
        "        self.head = nn.Sequential(\n",
        "            nn.Flatten(),\n",
        "            nn.Linear(n_features * d_model, d_model),\n",
        "            nn.GELU(),\n",
        "            nn.Dropout(dropout),\n",
        "            nn.Linear(d_model, pred_len),\n",
        "        )\n",
        "\n",
        "    def forward(self, x):\n",
        "        x = x.transpose(1, 2)\n",
        "        x = self.value_embedding(x)\n",
        "        x = self.encoder(x)\n",
        "        x = self.norm(x)\n",
        "        return self.head(x)\n",
        "\n",
        "class CompetitionAlignedRMSELoss(nn.Module):\n",
        "    def __init__(self, threshold=1.5, high_weight=1.0, low_weight=0.3):\n",
        "        super().__init__()\n",
        "        self.threshold = threshold\n",
        "        self.high_weight = high_weight\n",
        "        self.low_weight = low_weight\n",
        "\n",
        "    def forward(self, pred, target, origin_hs):\n",
        "        diff_sq = (pred - target) ** 2\n",
        "        weights = torch.where(\n",
        "            origin_hs >= self.threshold,\n",
        "            torch.tensor(self.high_weight, device=pred.device),\n",
        "            torch.tensor(self.low_weight, device=pred.device)\n",
        "        ).unsqueeze(-1)\n",
        "        weighted_mse = torch.sum(weights * diff_sq) / torch.sum(weights)\n",
        "        return torch.sqrt(weighted_mse + 1e-6)\n",
        "\n",
        "def rmse(y_true, y_pred):\n",
        "    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))\n",
        "\n",
        "def competition_like_rmse(y_true, y_pred, origin_hs):\n",
        "    mask = origin_hs >= 1.5\n",
        "    if mask.sum() == 0: return np.nan, 0\n",
        "    return rmse(y_true[mask], y_pred[mask]), int(mask.sum())\n",
        "\n",
        "def competition_aligned_rmse(y_true, y_pred, origin_hs, threshold=1.5, high_weight=1.0, low_weight=0.3):\n",
        "    diff_sq = (y_pred - y_true) ** 2\n",
        "    weights = np.where(origin_hs >= threshold, high_weight, low_weight)[:, None]\n",
        "    return float(np.sqrt(np.sum(weights * diff_sq) / np.sum(weights) + 1e-6))\n",
        "\n",
        "def select_78h_separated_indices(times, station_ids, origin_hs, min_hours=78):\n",
        "    selected = []\n",
        "    times = pd.to_datetime(times)\n",
        "    for sid in np.unique(station_ids):\n",
        "        idx = np.where((station_ids == sid) & (origin_hs >= 1.5))[0]\n",
        "        idx = idx[np.argsort(times[idx])]\n",
        "        last_time = None\n",
        "        for i in idx:\n",
        "            t = times[i]\n",
        "            if last_time is None or (t - last_time) >= pd.Timedelta(hours=min_hours):\n",
        "                selected.append(i)\n",
        "                last_time = t\n",
        "    return np.asarray(selected, dtype=int)\n",
        "\n",
        "def evaluate_model(model, loader):\n",
        "    model.eval()\n",
        "    preds, ys, origins = [], [], []\n",
        "    with torch.no_grad():\n",
        "        for X, y, origin_hs, _ in loader:\n",
        "            X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)\n",
        "            preds.append(model(X).cpu().numpy())\n",
        "            ys.append(y.cpu().numpy())\n",
        "            origins.append(origin_hs.numpy())\n",
        "    y_true = np.concatenate(ys)\n",
        "    y_pred = np.concatenate(preds)\n",
        "    origin_hs = np.concatenate(origins)\n",
        "    overall = rmse(y_true, y_pred)\n",
        "    comp, comp_n = competition_like_rmse(y_true, y_pred, origin_hs)\n",
        "    aligned = competition_aligned_rmse(y_true, y_pred, origin_hs)\n",
        "    lead_dict = {f\"rmse_{h}h\": rmse(y_true[:, i], y_pred[:, i]) for i, h in enumerate(LEAD_HOURS)}\n",
        "    return {\"overall_rmse\": overall, \"comp_rmse\": comp, \"competition_aligned_rmse\": aligned, \"comp_valid_n\": comp_n, **lead_dict}, y_true, y_pred\n",
        "\n",
        "train_loader = make_loader(train_samples, batch_size=FINAL_PARAMS[\"batch_size\"], shuffle=True)\n",
        "valid_loader = make_loader(valid_samples, batch_size=FINAL_PARAMS[\"batch_size\"], shuffle=False)\n",
        "\n",
        "model = ITransformer(\n",
        "    seq_len=INPUT_LEN, n_features=train_samples[\"n_features\"], pred_len=N_TARGETS,\n",
        "    d_model=FINAL_PARAMS[\"d_model\"], n_heads=FINAL_PARAMS[\"n_heads\"], e_layers=FINAL_PARAMS[\"e_layers\"],\n",
        "    dropout=FINAL_PARAMS[\"dropout\"], d_ff=FINAL_PARAMS[\"d_ff\"]\n",
        ").to(DEVICE)\n",
        "\n",
        "optimizer = torch.optim.AdamW(model.parameters(), lr=FINAL_PARAMS[\"lr\"], weight_decay=FINAL_PARAMS[\"weight_decay\"])\n",
        "criterion = CompetitionAlignedRMSELoss(threshold=1.5, high_weight=1.0, low_weight=0.3)\n",
        "\n",
        "best_state = None\n",
        "best_score = np.inf\n",
        "wait = 0\n",
        "\n",
        "for epoch in range(1, FINAL_EPOCHS + 1):\n",
        "    model.train()\n",
        "    train_losses = []\n",
        "    for X, y, origin_hs, _ in train_loader:\n",
        "        X, y, origin_hs = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True), origin_hs.to(DEVICE, non_blocking=True)\n",
        "        optimizer.zero_grad(set_to_none=True)\n",
        "        loss = criterion(model(X), y, origin_hs)\n",
        "        loss.backward()\n",
        "        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)\n",
        "        optimizer.step()\n",
        "        train_losses.append(loss.item())\n",
        "\n",
        "    metrics, _, _ = evaluate_model(model, valid_loader)\n",
        "    score = metrics[\"competition_aligned_rmse\"]\n",
        "    print(f\"Epoch {epoch:02d} | train_loss={np.mean(train_losses):.5f} | overall_rmse={metrics['overall_rmse']:.5f} | comp_rmse={metrics['comp_rmse']:.5f} | aligned={score:.5f}\")\n",
        "\n",
        "    if score < best_score:\n",
        "        best_score = score\n",
        "        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n",
        "        wait = 0\n",
        "    else:\n",
        "        wait += 1\n",
        "    if wait >= FINAL_PATIENCE:\n",
        "        print(f\"Early stopping triggered at epoch {epoch}\")\n",
        "        break\n",
        "\n",
        "if best_state is not None:\n",
        "    model.load_state_dict(best_state)\n",
        "\n",
        "final_metrics, y_true, y_pred = evaluate_model(model, valid_loader)\n",
        "sep_idx = select_78h_separated_indices(valid_samples[\"origin_time\"], valid_samples[\"station_id\"], valid_samples[\"origin_hs\"], min_hours=78)\n",
        "final_metrics[\"exact_78h_comp_rmse\"] = rmse(y_true[sep_idx], y_pred[sep_idx]) if len(sep_idx) > 0 else np.nan\n",
        "final_metrics[\"exact_78h_n\"] = len(sep_idx)\n",
        "\n",
        "print(\"\\n===== FINAL VALIDATION METRICS =====\")\n",
        "display(pd.Series(final_metrics).to_frame(\"value\"))\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": None,
      "metadata": {},
      "outputs": [],
      "source": [
        "# ============================================================\n",
        "# 6. STEP 5: TEST INFERENCE & SUBMISSION CREATION\n",
        "# ============================================================\n",
        "print(\"\\n\" + \"=\" * 70)\n",
        "print(\"[Step 5/5] Generating final submission from test_context.parquet...\")\n",
        "print(\"=\" * 70)\n",
        "\n",
        "BASE_FALLBACK_COLS = [\"tp\", \"wspd\", \"gust\", \"wdir\", \"wvdir\", \"airt\", \"relh\", \"caph\"]\n",
        "station_medians = train_final_physics.groupby(STATION_COL)[BASE_FALLBACK_COLS].median()\n",
        "global_medians = train_final_physics[BASE_FALLBACK_COLS].median()\n",
        "\n",
        "ratio_df = train_final_physics[train_final_physics[\"hs\"].notna() & train_final_physics[\"hmax\"].notna() & (train_final_physics[\"hs\"] > 0)].copy()\n",
        "ratio_df[\"hmax_hs_ratio\"] = ratio_df[\"hmax\"] / ratio_df[\"hs\"]\n",
        "ratio_df = ratio_df[ratio_df[\"hmax_hs_ratio\"].between(1.0, 3.0)]\n",
        "station_hmax_ratio = ratio_df.groupby(STATION_COL)[\"hmax_hs_ratio\"].median()\n",
        "global_hmax_ratio = float(ratio_df[\"hmax_hs_ratio\"].median())\n",
        "\n",
        "def get_station_median(station, col):\n",
        "    value = np.nan\n",
        "    if station in station_medians.index and col in station_medians.columns:\n",
        "        value = station_medians.loc[station, col]\n",
        "    if not np.isfinite(value):\n",
        "        value = global_medians[col]\n",
        "    return float(value)\n",
        "\n",
        "def get_hmax_ratio(station):\n",
        "    if station in station_hmax_ratio.index:\n",
        "        val = station_hmax_ratio.loc[station]\n",
        "        if np.isfinite(val):\n",
        "            return float(val)\n",
        "    return global_hmax_ratio\n",
        "\n",
        "def add_test_features_exp04(context):\n",
        "    feature_frames = []\n",
        "    for case_id, group in context.groupby(\"case_id\", sort=False):\n",
        "        g = group.sort_values(\"step_minute\").copy()\n",
        "        st = g[STATION_COL].iloc[0]\n",
        "\n",
        "        for col in [\"hs\", \"tp\", \"hmax\"]:\n",
        "            g.loc[g[col] <= 0, col] = np.nan\n",
        "        g.loc[g[\"wspd\"] < 0, \"wspd\"] = np.nan\n",
        "        g.loc[g[\"gust\"] < 0, \"gust\"] = np.nan\n",
        "        g.loc[~g[\"relh\"].between(0, 100), \"relh\"] = np.nan\n",
        "        g.loc[~g[\"caph\"].between(950, 1050), \"caph\"] = np.nan\n",
        "\n",
        "        num_cols = [\"hs\", \"tp\", \"hmax\", \"wspd\", \"gust\", \"airt\", \"relh\", \"caph\"]\n",
        "        g[num_cols] = g[num_cols].interpolate(method=\"linear\", limit_direction=\"both\").ffill().bfill()\n",
        "\n",
        "        for col in [\"wdir\", \"wvdir\"]:\n",
        "            g[col] = g[col].ffill().bfill()\n",
        "            if g[col].isna().any():\n",
        "                g[col] = g[col].fillna(get_station_median(st, col))\n",
        "        g[\"wdir\"] %= 360.0\n",
        "        g[\"wvdir\"] %= 360.0\n",
        "\n",
        "        if g[\"hs\"].isna().any():\n",
        "            g[\"hs\"] = g[\"hs\"].ffill().bfill()\n",
        "\n",
        "        hmax_missing = g[\"hmax\"].isna() & g[\"hs\"].notna()\n",
        "        if hmax_missing.any():\n",
        "            g.loc[hmax_missing, \"hmax\"] = g.loc[hmax_missing, \"hs\"] * get_hmax_ratio(st)\n",
        "\n",
        "        for col in [\"tp\", \"wspd\", \"gust\", \"airt\", \"relh\", \"caph\"]:\n",
        "            if g[col].isna().any():\n",
        "                g[col] = g[col].fillna(get_station_median(st, col))\n",
        "\n",
        "        g[\"hs\"] = g[\"hs\"].clip(lower=0.01)\n",
        "        g[\"tp\"] = g[\"tp\"].clip(lower=0.01)\n",
        "        g[\"wspd\"] = g[\"wspd\"].clip(lower=0.0)\n",
        "        g[\"gust\"] = g[\"gust\"].clip(lower=0.0)\n",
        "        g[\"hmax\"] = np.maximum(g[\"hmax\"], g[\"hs\"])\n",
        "        g[\"gust\"] = np.maximum(g[\"gust\"], g[\"wspd\"])\n",
        "\n",
        "        wdir_rad = np.deg2rad(g[\"wdir\"])\n",
        "        wvdir_rad = np.deg2rad(g[\"wvdir\"])\n",
        "        g[\"u_wind\"] = g[\"wspd\"] * np.sin(wdir_rad)\n",
        "        g[\"v_wind\"] = g[\"wspd\"] * np.cos(wdir_rad)\n",
        "        g[\"u_wave\"] = g[\"hs\"] * np.sin(wvdir_rad)\n",
        "        g[\"v_wave\"] = g[\"hs\"] * np.cos(wvdir_rad)\n",
        "\n",
        "        diff = (g[\"wdir\"] - g[\"wvdir\"] + 180) % 360 - 180\n",
        "        g[\"wind_wave_diff\"] = np.abs(diff)\n",
        "        g[\"wind_wave_alignment\"] = np.cos(np.deg2rad(diff))\n",
        "\n",
        "        g[\"hs_diff_1h\"] = g[\"hs\"] - g[\"hs\"].shift(6)\n",
        "        g[\"hs_diff_3h\"] = g[\"hs\"] - g[\"hs\"].shift(18)\n",
        "        g[\"hs_mean_6h\"] = g[\"hs\"].rolling(36, min_periods=1).mean()\n",
        "        g[\"hs_mean_12h\"] = g[\"hs\"].rolling(72, min_periods=1).mean()\n",
        "        g[\"hs_max_6h\"] = g[\"hs\"].rolling(36, min_periods=1).max()\n",
        "        g[\"hs_max_12h\"] = g[\"hs\"].rolling(72, min_periods=1).max()\n",
        "\n",
        "        g[\"wspd_mean_6h\"] = g[\"wspd\"].rolling(36, min_periods=1).mean()\n",
        "        g[\"wspd_mean_12h\"] = g[\"wspd\"].rolling(72, min_periods=1).mean()\n",
        "        g[\"gust_max_6h\"] = g[\"gust\"].rolling(36, min_periods=1).max()\n",
        "        g[\"gust_max_12h\"] = g[\"gust\"].rolling(72, min_periods=1).max()\n",
        "        g[\"gust_minus_wspd\"] = g[\"gust\"] - g[\"wspd\"]\n",
        "\n",
        "        g[\"caph_change_3h\"] = g[\"caph\"] - g[\"caph\"].shift(18)\n",
        "        g[\"caph_change_6h\"] = g[\"caph\"] - g[\"caph\"].shift(36)\n",
        "        g[\"caph_change_12h\"] = g[\"caph\"] - g[\"caph\"].shift(72)\n",
        "\n",
        "        wl = 1.56 * (g[\"tp\"] ** 2)\n",
        "        g[\"wave_steepness\"] = g[\"hs\"] / np.maximum(wl, 1.0)\n",
        "        g[\"wave_energy\"] = g[\"hs\"] ** 2\n",
        "        g[\"effective_wind_forcing\"] = (g[\"wspd\"] ** 2) * g[\"wind_wave_alignment\"]\n",
        "\n",
        "        g = g.replace([np.inf, -np.inf], np.nan)\n",
        "        g[BEST_FEATURES] = g[BEST_FEATURES].bfill().ffill()\n",
        "        feature_frames.append(g)\n",
        "    return pd.concat(feature_frames, ignore_index=True)\n",
        "\n",
        "test_context = pd.read_parquet(TEST_CONTEXT_PATH)\n",
        "test_index = pd.read_csv(TEST_INDEX_PATH)\n",
        "test_features = add_test_features_exp04(test_context)\n",
        "\n",
        "case_order = test_index[\"case_id\"].drop_duplicates().tolist()\n",
        "windows = []\n",
        "for case_id in case_order:\n",
        "    grp = test_features[test_features[\"case_id\"] == case_id].sort_values(\"step_minute\")\n",
        "    windows.append(grp[BEST_FEATURES].to_numpy(dtype=np.float32))\n",
        "\n",
        "X_test_raw = np.stack(windows)\n",
        "X_test = scaler.transform(X_test_raw.reshape(-1, len(BEST_FEATURES))).reshape(X_test_raw.shape).astype(np.float32)\n",
        "\n",
        "model.eval()\n",
        "pred_batches = []\n",
        "INFER_BATCH_SIZE = 128\n",
        "with torch.no_grad():\n",
        "    for start in range(0, len(X_test), INFER_BATCH_SIZE):\n",
        "        xb = torch.from_numpy(X_test[start:start + INFER_BATCH_SIZE]).to(DEVICE)\n",
        "        pred_batches.append(model(xb).detach().cpu().numpy())\n",
        "\n",
        "preds = np.concatenate(pred_batches, axis=0)\n",
        "\n",
        "pred_rows = []\n",
        "for case_id, row in zip(case_order, preds):\n",
        "    for lead_h, val in zip(LEAD_HOURS, row):\n",
        "        pred_rows.append({\"case_id\": case_id, \"lead_h\": lead_h, \"hs_pred\": float(val)})\n",
        "\n",
        "submission = test_index.merge(pd.DataFrame(pred_rows), on=[\"case_id\", \"lead_h\"], how=\"left\", validate=\"one_to_one\")\n",
        "submission[\"hs_pred\"] = submission[\"hs_pred\"].clip(lower=0.0, upper=30.0)\n",
        "submission.to_csv(SUBMISSION_PATH, index=False, encoding=\"utf-8\")\n",
        "\n",
        "print(\"=\" * 80)\n",
        "print(f\"ALL DONE! EXP04 SUBMISSION FILE SAVED TO: {SUBMISSION_PATH.resolve()}\")\n",
        "print(\"=\" * 80)\n",
        "display(submission.head(12))\n"
      ]
    }
  ],
  "metadata": {
    "accelerator": "GPU",
    "colab": {
      "gpuType": "T4",
      "provenance": []
    },
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "version": "3.10.12"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}

{'cells': [{'cell_type': 'markdown',
   'metadata': {},
   'source': ['# EXP04 Full Reproduction Pipeline (From Raw Data to Submission)\n',
    '\n',
    '대회 원본 관측 데이터(`train_atmos.csv`, `train_wave.csv`)로부터 시작하여, 전처리 및 물리 피처 엔지니어링, EXP04 최종 하이퍼파라미터 모델 학습, 테스트 추론 및 제출 파일 생성까지 한 번에 수행하는 전체 재현 노트북입니다.\n',
    '\n',
    '### 파이프라인 단계\n',
    '1. **Raw Merge & Grid Alignment (`train_v2`)**: 해양/기상 원본 결합 및 3개 기지 10분 정규 격자(236,304행) 구축\n',
    '2. **Physics Imputation (`train_v4` Stage 1 & 2)**: 1스텝 보간, Rayleigh 파고 역학, Wilson 풍파 평형, 기압/온습도 미기후 물리 대치\n',
    '3. **Physics Feature Generation (`train_v4` Stage 3)**: EXP04 최적 31개 피처(`ALL_FEATURES`) 산출\n',
    '4. **iTransformer Final Model Training**: Optuna Trial 25 최적 하이퍼파라미터 및 `CompetitionAlignedRMSELoss` 기반 학습\n',
    '5. **Test Inference & Submission**: `test_context.parquet` 결측 방어 처리 후 6개 리드타임 예측 및 `submission_exp04.csv` 생성']},
  {'cell_type': 'code',
   'execution_count': None,
   'metadata': {},
   'outputs': [],
   'source': ['# ========